# Project 2: IMDb Top 250 Movies Analysis
## Data Cleaning, Preprocessing & 10 Analytical Tasks
* **Dataset:** `movies.csv` (IMDb Top 250 Movies)
* **Libraries:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `wordcloud`


In [ ]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style='whitegrid')


## Preparation: Load and Clean the Data
1. Load `movies.csv`
2. Convert `imbd_votes` to integer (remove commas)
3. Convert `duration` text (e.g. `"2h 22m"`, `"3h"`, `"45m"`) to total minutes (`duration_in_min`)
4. Handle missing values and check for duplicate records


In [ ]:
# Load the dataset
df = pd.read_csv('movies.csv')
print("Initial dataset shape:", df.shape)
df.head(3)


In [ ]:
# Step 2: Convert imbd_votes to integer
df['imbd_votes'] = df['imbd_votes'].str.replace(',', '').astype(int)

# Step 3: Convert duration into minutes
def duration_to_min(str_value):
    hours = 0
    minutes = 0
    if 'h' in str_value:
        hours = int(str_value.split('h')[0])
    if 'm' in str_value:
        minutes = int(str_value.split('h')[-1].split('m')[0])
    return hours * 60 + minutes

df['duration_in_min'] = df['duration'].apply(duration_to_min)

# Step 4: Handle missing values and duplicates
print("Missing values before cleaning:
", df.isna().sum()[df.isna().sum() > 0])
df.dropna(inplace=True)

print("Duplicates count:", df.duplicated().sum())
print("Cleaned dataset shape:", df.shape)


---
### Q1: Plot a graph to show the top 5 movies based on the number of votes.


In [ ]:
top_5_movies = df.sort_values(by='imbd_votes', ascending=False).head(5)
print(top_5_movies[['title', 'genre', 'duration', 'imbd_votes']])

plt.figure(figsize=(12, 5))
sns.barplot(y='title', x='imbd_votes', data=top_5_movies, palette='Blues_r')
plt.title('Top 5 Highest Voted Films on IMDb', fontsize=16, fontweight='bold')
plt.xlabel('Number of Votes', fontsize=12)
plt.ylabel('Movie Title', fontsize=12)
plt.show()


---
### Q2: Find which movie has the maximum number of votes, its genre and its duration, and make a word cloud of its `review_content`.


In [ ]:
max_vote_movie = df.loc[df['imbd_votes'].idxmax(), ['title', 'genre', 'duration', 'duration_in_min', 'imbd_votes', 'review_content']]

print("Movie with Maximum Votes:")
print("  Title:   ", max_vote_movie['title'])
print("  Genre:   ", max_vote_movie['genre'])
print("  Duration:", max_vote_movie['duration'], f"({max_vote_movie['duration_in_min']} min)")
print("  Votes:   ", f"{max_vote_movie['imbd_votes']:,}")

# Word cloud of its review content
top_voted_review = max_vote_movie['review_content']

plt.figure(figsize=(14, 7))
word_cloud = WordCloud(stopwords=STOPWORDS, colormap='plasma',
                       width=800, height=400,
                       background_color='black').generate(top_voted_review)
plt.imshow(word_cloud, interpolation='bilinear')
plt.axis('off')
plt.title(f"Review Word Cloud: {max_vote_movie['title']}", fontsize=16, fontweight='bold')
plt.show()


---
### Q3: Find the movie with the maximum number of votes in each genre.


In [ ]:
# Highest voted movie per genre combination
max_votes_each_genre = df.loc[df.groupby('genre')['imbd_votes'].idxmax(), ['title', 'genre', 'imbd_votes']]
max_votes_each_genre = max_votes_each_genre.sort_values(by='imbd_votes', ascending=False).reset_index(drop=True)

print(f"Total unique genre categories: {len(max_votes_each_genre)}")
max_votes_each_genre.head(15)


---
### Q4: Find which movie has the minimum number of votes, its genre and its duration.


In [ ]:
min_vote_movie = df.loc[df['imbd_votes'].idxmin(), ['title', 'genre', 'duration', 'duration_in_min', 'imbd_votes']]

print("Movie with Minimum Votes:")
print("  Title:   ", min_vote_movie['title'])
print("  Genre:   ", min_vote_movie['genre'])
print("  Duration:", min_vote_movie['duration'], f"({min_vote_movie['duration_in_min']} min)")
print("  Votes:   ", f"{min_vote_movie['imbd_votes']:,}")


---
### Q5: Find the average movie duration in each genre.


In [ ]:
# Calculate average duration per genre
avg_duration_genre = df.groupby('genre', as_index=False)[['duration_in_min']].mean()
avg_duration_genre.rename(columns={'duration_in_min': 'avg_duration_min'}, inplace=True)
avg_duration_genre['avg_duration_hr'] = (avg_duration_genre['avg_duration_min'] / 60).round(2)
avg_duration_genre['avg_duration_min'] = avg_duration_genre['avg_duration_min'].round(1)

# Display top 15 genres by duration
avg_duration_genre.sort_values(by='avg_duration_min', ascending=False).reset_index(drop=True).head(15)


---
### Q6 (Bonus): Create a word cloud of the storylines for the most common genres.


In [ ]:
top_genres = df['genre'].value_counts().head(3).index

for genre in top_genres:
    text = ' '.join(df.loc[df['genre'] == genre, 'storyline'])
    plt.figure(figsize=(12, 5))
    word_cloud = WordCloud(stopwords=STOPWORDS, colormap='magma',
                           width=800, height=350,
                           background_color='black').generate(text)
    plt.imshow(word_cloud, interpolation='bilinear')
    plt.title(f'Storyline Word Cloud: {genre}', fontsize=15, fontweight='bold')
    plt.axis('off')
    plt.show()


---
### Q7: What is the average IMDb rating by decade of movie release?


In [ ]:
df['decade'] = (df['year'] // 10) * 10
decade_ratings = df.groupby('decade')['imbd_rating'].agg(['mean', 'count']).reset_index()
decade_ratings.columns = ['Decade', 'Average_Rating', 'Movie_Count']
decade_ratings['Average_Rating'] = decade_ratings['Average_Rating'].round(2)

plt.figure(figsize=(10, 5))
sns.lineplot(data=decade_ratings, x='Decade', y='Average_Rating', marker='o', color='crimson', linewidth=2.5)
plt.title('Average IMDb Rating by Release Decade', fontsize=15, fontweight='bold')
plt.xlabel('Decade', fontsize=12)
plt.ylabel('Average IMDb Rating', fontsize=12)
plt.show()

decade_ratings


---
### Q8: Which directors have directed the most movies in the IMDb Top 250?


In [ ]:
top_directors = df['director_name'].value_counts().head(10).reset_index()
top_directors.columns = ['Director', 'Movie_Count']

plt.figure(figsize=(12, 5))
sns.barplot(data=top_directors, y='Director', x='Movie_Count', palette='crest')
plt.title('Top 10 Directors with Most Movies in IMDb Top 250', fontsize=15, fontweight='bold')
plt.xlabel('Number of Movies in Top 250', fontsize=12)
plt.ylabel('Director Name', fontsize=12)
plt.show()

top_directors


---
### Q9: Find the top 5 longest and top 5 shortest movies in the dataset.


In [ ]:
# Top 5 longest movies
longest = df.sort_values(by='duration_in_min', ascending=False)[['title', 'year', 'genre', 'duration', 'duration_in_min']].head(5)
print("=== Top 5 Longest Movies ===")
print(longest.to_string(index=False))

# Top 5 shortest movies
shortest = df.sort_values(by='duration_in_min', ascending=True)[['title', 'year', 'genre', 'duration', 'duration_in_min']].head(5)
print("
=== Top 5 Shortest Movies ===")
print(shortest.to_string(index=False))


---
### Q10: What is the distribution of movie certificates (age ratings) in the Top 250?


In [ ]:
cert_counts = df['certificate'].value_counts().reset_index()
cert_counts.columns = ['Certificate', 'Count']

plt.figure(figsize=(11, 5))
sns.barplot(data=cert_counts, x='Certificate', y='Count', palette='viridis')
plt.title('Distribution of Movie Certificates in IMDb Top 250', fontsize=15, fontweight='bold')
plt.xlabel('Certificate Rating', fontsize=12)
plt.ylabel('Number of Movies', fontsize=12)
plt.xticks(rotation=45)
plt.show()

cert_counts
